# Building Vision Datasets

This notebook walks through the full pipeline for building vision/image datasets:

1. **Scraping** images from the web using search providers
2. **Cleaning** downloaded images — deduplication by perceptual hash and vision-based filtering with Ollama
3. **Transforming** raw downloads into structured classification records
4. **Formatting** for vision ML tasks in industry-standard formats

### Supported Vision Formats

| Format | Use Case |
|--------|----------|
| **ImageFolder** | HuggingFace standard for image classification |
| **COCO** | Object detection and captioning |
| **YOLO** | Compact object detection format |
| **LLaVA** | Vision-language model training |
| **COCO-Seg** | Instance segmentation |
| **YOLO-Seg** | Segmentation in YOLO format |
| **CSV-Images** | Tabular metadata with image paths |

In [ ]:
!npm start -- scrape --search "bird species" --download --formats image --search-count 3 -y

In [ ]:
import json
import os
from pathlib import Path

# Find the most recent task folder
output_dir = Path("../output")
task_folders = sorted(output_dir.glob("task_*"), key=os.path.getmtime, reverse=True)

if not task_folders:
    print("No task folders found. Run the scrape cell above first.")
else:
    task_folder = task_folders[0]
    manifest_path = task_folder / "downloads" / "manifest.json"

    if manifest_path.exists():
        with open(manifest_path) as f:
            manifest = json.load(f)

        print(f"Task folder: {task_folder.name}")
        print(f"Total downloads: {len(manifest)}")

        # Count file types
        extensions = {}
        for entry in manifest:
            ext = Path(entry.get("filename", "")).suffix.lower()
            extensions[ext] = extensions.get(ext, 0) + 1

        print("\nFile types:")
        for ext, count in sorted(extensions.items(), key=lambda x: -x[1]):
            print(f"  {ext or '(no ext)'}: {count}")
    else:
        print(f"Manifest not found at {manifest_path}")

## Vision Cleaning

Before building a dataset, it's important to clean the downloaded images:

- **Perceptual hash deduplication** — Detects near-duplicate images even if they differ in resolution, compression, or minor edits. Uses pHash to generate fingerprints and removes images that are too similar.
- **Vision filtering with Ollama** — Uses a vision-capable LLM to verify that images actually match the intended content, filtering out irrelevant or low-quality results.

In [ ]:
!npm start -- clean -i ../output/task_*/downloads --dedupe-images -y

In [ ]:
!npm start -- transform -i ../output/task_* -t image-classification --target "bird species" -o ../output/nb_bird_classified.json

## Vision Format Options

The `format` command supports several vision-specific output formats:

- **ImageFolder** — The HuggingFace standard for image classification. Organizes images into subdirectories by label, with an optional `metadata.jsonl` file. Directly loadable with `datasets.load_dataset("imagefolder", ...)`.

- **COCO** — The Common Objects in Context format, widely used for object detection and image captioning. Produces a JSON annotation file with image metadata, categories, and annotations.

- **YOLO** — A compact format for object detection. Each image gets a corresponding `.txt` file with normalized bounding box coordinates. Includes a `data.yaml` config for training.

- **LLaVA** — Format for training vision-language models. Pairs images with conversational Q&A in a JSON structure, suitable for instruction-tuned multimodal models.

In [ ]:
!npm start -- format -i ../output/nb_bird_classified.json -f imagefolder -o ../output/nb_imagefolder/ --copy-media

In [ ]:
!npm start -- format -i ../output/nb_bird_classified.json -f coco -o ../output/nb_coco/

In [ ]:
from pathlib import Path
import os

def print_tree(directory, prefix="", max_depth=3, current_depth=0):
    """Print a directory tree up to max_depth."""
    if current_depth >= max_depth:
        return
    path = Path(directory)
    if not path.exists():
        print(f"{prefix}{path.name}/ (not found)")
        return
    entries = sorted(path.iterdir())
    for i, entry in enumerate(entries):
        connector = "+-" if i < len(entries) - 1 else "\\-"
        if entry.is_dir():
            file_count = sum(1 for _ in entry.rglob("*") if _.is_file())
            print(f"{prefix}{connector} {entry.name}/ ({file_count} files)")
            next_prefix = prefix + ("|  " if i < len(entries) - 1 else "   ")
            print_tree(entry, next_prefix, max_depth, current_depth + 1)
        else:
            size = entry.stat().st_size
            print(f"{prefix}{connector} {entry.name} ({size:,} bytes)")

print("=== ImageFolder Output ===")
print_tree("../output/nb_imagefolder")

print("\n=== COCO Output ===")
print_tree("../output/nb_coco")

In [ ]:
# Optional: Load with HuggingFace datasets library
# Requires: pip install datasets Pillow

try:
    from datasets import load_dataset

    ds = load_dataset("imagefolder", data_dir="../output/nb_imagefolder")
    print(f"Dataset loaded successfully!")
    print(f"Splits: {list(ds.keys())}")
    for split_name, split_data in ds.items():
        print(f"  {split_name}: {len(split_data)} examples")
        print(f"  Features: {split_data.features}")
    print(f"\nFirst example: {ds[list(ds.keys())[0]][0]}")

except ImportError:
    print("HuggingFace datasets library not installed.")
    print("Install with: pip install datasets Pillow")
    print("\nSkipping this cell — the exported files are still valid.")
except Exception as e:
    print(f"Error loading dataset: {e}")

In [ ]:
import shutil
from pathlib import Path

# Cleanup notebook outputs
cleanup_paths = [
    "../output/nb_bird_classified.json",
    "../output/nb_imagefolder",
    "../output/nb_coco",
]

for p in cleanup_paths:
    path = Path(p)
    if path.is_dir():
        shutil.rmtree(path)
        print(f"Removed directory: {path}")
    elif path.is_file():
        path.unlink()
        print(f"Removed file: {path}")
    else:
        print(f"Not found (skipped): {path}")

print("\nCleanup complete.")